# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/7ayder-99/flyrank_internship/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

*The research question and the decision it supports.*

Research Question:
Can a data-driven ranking model prioritize content pages for review better than a simple rule-based refresh score, using observable search-performance and content signals?

Decision Supported:
The goal is to help content and SEO teams decide which pages should be reviewed first when resources are limited. The ranking provides a decision-support queue that can prioritize pages for actions such as refresh, expansion, monitoring, or protection. It does not claim that refreshing a page will cause traffic growth; it is intended to make the review process more focused and evidence-based.

## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

The analysis uses the FlyRank internship dataset and focuses on content-page search performance across two monthly windows.

Release: The internship dataset release provided for the capstone.
Tables used: Content/page-level performance data joined with the content dimension table to obtain content update dates.
Date windows: February 2026 and March 2026 were used as the main performance windows. Content freshness was calculated relative to March 31, 2026 using content_updated_date.
Unit of analysis: One content page identified by client_hash_id and content_hash_id.
Merged dataset: 303,572 pages were present in both monthly windows and were used for the baseline ranking analysis.
Excluded: Client names, domains, URLs, private search queries, credentials, and raw identifying exports were excluded from the analysis and final artifacts to keep the work public-safe.

The analysis uses only observable historical and content-level signals available within the dataset. No future performance window was used in the baseline score, and no causal claims are made about the effect of refreshing content.

## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

The analysis treats refresh prioritization as a ranking/scoring problem, where the goal is to identify pages that deserve review rather than to predict a binary outcome.

Assumptions

The baseline assumes that a page is a stronger refresh candidate when it:

Has not been updated recently.
Had meaningful historical search visibility.
Experienced a decline in impressions between February and March 2026.

These assumptions are treated as hypotheses to test, not as established causal relationships.

Features

The main signals used in the baseline were:

days_since_update: days between the content update date and March 31, 2026.
impressions_feb: historical search impressions.
impressions_mar: March search impressions.
declining: whether March impressions were lower than February impressions.
stale: whether days_since_update >= 180.
visible: whether impressions_feb >= 100.

Content and client identifiers were used for grouping and joining, not as predictive signals.

Label Definition

The analysis uses a decision proxy rather than an observed business outcome. A page is considered a baseline review candidate when all three conditions are satisfied:

stale + historically visible + declining

The resulting action score is:

action_score =
    impressions_feb
    if stale, visible, and declining
    else 0

This score represents the amount of historical visibility potentially affected by the observed decline. It does not represent predicted traffic recovery.

Baseline

The baseline is a simple rule-based refresh score. Pages satisfying all three conditions receive the action:

review_for_refresh

and are ranked by action_score. All other pages receive:

monitor

The baseline produced a ranked queue of 303,572 pages, with 19 pages flagged for refresh review.

Validation Design

The baseline signals were checked using grouped comparisons before treating them as useful prioritization signals. In particular, content freshness was evaluated across staleness tiers, while CTR was examined across average-position tiers.

The staleness analysis produced an unexpected result: recently updated pages showed higher observed decline rates than older pages. Therefore, the staleness assumption was treated as weak rather than confirmed.

The position/CTR analysis showed the expected directional pattern: mean CTR decreased as average position became worse.

Leakage Checks

Potential leakage was considered explicitly. Fields such as trend_direction and trend_pct were not used as predictive features because they directly describe the performance trend being analyzed.

The baseline score uses February and March observations and the fixed March 31, 2026 analysis date. No post-March performance window was used in the baseline calculation.

The resulting ranking should therefore be interpreted as observational decision support, not as a causal model or a guarantee that refreshing a selected page will improve future performance.

## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*



The baseline provides a simple reference point for evaluating a data-driven ranking approach. Both approaches should be evaluated on the **same validation split** using the same ranking metric, with **Precision@50** as the primary decision metric.

At the current stage, the rule-based baseline has been fully implemented and validated. It ranked 303,572 pages and flagged 19 pages for refresh review.

| Approach                    | Evaluation setup      |   Precision@50 |
| --------------------------- | --------------------- | -------------: |
| Rule-based refresh baseline | Same validation split | To be measured |
| Data-driven ranking model   | Same validation split | To be measured |

The baseline is intentionally simple: it prioritizes pages that are stale, historically visible, and declining in impressions. Signal validation showed that the staleness assumption did not behave as expected in the observed data, while the position/CTR relationship showed the expected directional pattern.

Therefore, the final comparison will determine whether the data-driven model provides better top-50 prioritization than the rule-based baseline. Until that evaluation is run on the same split, no performance advantage is claimed for the model.


## 5. Limitations

*What this work cannot claim.*



This work is intended as a **decision-support ranking system**, not a causal or predictive guarantee.

* The analysis **cannot prove that refreshing a page will increase traffic, impressions, clicks, or engagement**. The relationships observed in the data are observational.
* The ranking does not establish **causal relationships between content updates and search performance**.
* The dataset covers a limited set of historical time windows, so the results may not generalize to future periods or different websites.
* The refresh label is a **proxy for prioritization**, not a directly observed business outcome such as successful traffic recovery after a refresh.
* The baseline depends on fixed thresholds, including the **180-day freshness threshold** and **100 historical impressions threshold**; different thresholds may produce different rankings.
* Some data-quality issues were observed, including content update dates occurring after the analysis date. These records require validation before being used for operational decisions.
* The model should **not replace human SEO or content judgment**. Final decisions should also consider content quality, search intent, business importance, seasonality, and other factors not fully represented in the dataset.
* The results should therefore be interpreted as **measured and directional evidence for prioritization**, rather than proof of future performance.


## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

## 6. Ranked Recommendations

The analysis produces a practical action playbook for prioritizing content review under limited resources.

### 1. Review stale, visible, and declining pages first

Prioritize pages that meet all three baseline conditions:

* At least **180 days since the last content update**.
* At least **100 historical impressions**.
* Lower impressions in March 2026 than in February 2026.

These pages receive the highest baseline priority because they combine historical visibility with an observed decline.

### 2. Start with the highest historical-visibility pages

Within the refresh-review queue, prioritize pages with higher February impressions. This focuses limited review capacity on pages that previously had stronger search visibility.

### 3. Treat the ranking as a review queue, not an automatic refresh decision

The score should identify pages for **human review**. The content/SEO team should then determine whether the appropriate action is refresh, expansion, protection, or monitoring.

### 4. Do not use content age alone as a refresh signal

The freshness analysis did not support the original assumption that older pages are more likely to decline. In the observed data, newer pages showed higher decline rates, so staleness should be treated as only one part of the decision process.

### 5. Prioritize pages with stronger search-position opportunities

The position analysis showed a clear directional relationship between ranking position and CTR: pages in better positions had higher average CTR. This signal can be used as additional context when reviewing opportunities, but it should not be interpreted as proof that changing position will cause higher CTR.

### 6. Validate data-quality anomalies before acting

Records with inconsistent dates, such as content update dates occurring after the analysis date, should be checked before they influence operational decisions.

**Recommended workflow:**
**Rank → Review → Diagnose → Choose action → Monitor outcome**

This keeps the system focused on prioritization while leaving the final content decision to human experts.


## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*


The deployed paper should include the following charts and tables to make the analysis transparent, reproducible, and easy to interpret.

### 1. Data Coverage Table

A summary of the analyzed dataset, including:

* Number of pages analyzed: **303,572**
* Performance windows: **February 2026 and March 2026**
* Number of clients represented
* Main signals used in the analysis

### 2. Content Freshness vs. Decline Chart

A bar chart showing the observed decline rate across content freshness tiers:

* `<30d`
* `30–90d`
* `90–180d`
* `180–365d`

This chart should highlight that the observed relationship did **not** support the assumption that older content was more likely to decline.

### 3. Search Position vs. CTR Chart

A bar chart showing average CTR by ranking-position tier:

* `1–3`
* `4–10`
* `11–20`
* `21–50`
* `50+`

This provides visual evidence for the observed relationship between search position and CTR.

### 4. Baseline Action Queue Table

A ranked table showing the highest-priority pages produced by the rule-based baseline, including:

| Rank | Content ID               | Feb Impressions | Mar Impressions | Days Since Update | Action Score | Recommended Action |
| ---: | ------------------------ | --------------: | --------------: | ----------------: | -----------: | ------------------ |
|    1 | content_3af16a3c5dffb146 |           1,912 |               0 |               252 |        1,912 | Review for refresh |
|    2 | content_715dfb7ac57fc2f3 |           1,130 |               0 |               218 |        1,130 | Review for refresh |
|    3 | content_5c9e0961371c7f2d |             887 |               0 |               234 |          887 | Review for refresh |
|    4 | content_84d0bb5aaf292517 |             790 |               0 |               280 |          790 | Review for refresh |
|    5 | content_70537a712a8d554e |             543 |               0 |               222 |          543 | Review for refresh |

The full ranked queue is saved as `work/outputs/baseline_action_score.csv`.

### 5. Baseline Summary

A compact summary showing that the baseline evaluated **303,572 pages**, with **19 pages flagged for refresh review** and **303,553 pages assigned to monitoring**.

These artifacts provide the evidence needed to understand the data, validate the main signals, inspect the baseline output, and reproduce the decision-support workflow.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [ ] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [ ] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.
